# About This Notebook

Since this notebook shows vendor-specific extensions, I wont' bother to try to run it.

# SQL in Big Data Ecosystem (e.g., HiveQL)

In this code snippet, we are demonstrating the usage of SQL in a Big Data ecosystem, specifically using HiveQL. HiveQL is a variation of SQL that is used in Apache Hive, a data warehouse infrastructure built on top of Hadoop.

The code begins by creating a table called "employees" with columns for id, name, age, and salary. This table will be used to store employee data.

Next, we insert some sample data into the "employees" table using the `INSERT INTO` statement. Multiple rows can be inserted in a single statement by separating them with commas.

Finally, we query all the employees from the "employees" table using the `SELECT` statement. The `*` symbol is used to select all columns from the table.

When executed, the code will create the table, insert the sample data, and then print the result of the SELECT query, which should display the employee data.

Expected output:
```
1 | John Doe | 30 | 5000.00
2 | Jane Smith | 35 | 6000.00
3 | Mike Johnson | 40 | 7000.00
```

In [ ]:
-- Create a table to store employee data
CREATE TABLE employees (
    id INT,
    name VARCHAR(100),
    age INT,
    salary DECIMAL(10, 2)
);

-- Insert some sample data into the table
INSERT INTO employees (id, name, age, salary)
VALUES (1, 'John Doe', 30, 5000.00),
       (2, 'Jane Smith', 35, 6000.00),
       (3, 'Mike Johnson', 40, 7000.00);

-- Query all employees
SELECT * FROM employees;

# Vendor-specific SQL Extensions (e.g., T-SQL, PL/SQL)

This code snippet demonstrates some of the vendor-specific SQL extensions, namely T-SQL (used by Microsoft SQL Server) and PL/SQL (used by Oracle Database). 

In T-SQL, we showcase the declaration and usage of variables, the IF statement for conditional logic, and the creation and execution of a stored procedure.

Please note that PL/SQL examples are not included in this code snippet as it is specific to Oracle Database. However, similar concepts like variable declaration, conditional statements, and stored procedures exist in PL/SQL as well.

In [ ]:
-- This code snippet demonstrates vendor-specific SQL extensions, specifically T-SQL and PL/SQL.

-- T-SQL (Transact-SQL) is the proprietary extension to SQL used by Microsoft SQL Server.
-- It includes additional features and syntax that are not part of the SQL standard.

-- Example 1: Declaring a variable and printing its value
DECLARE @name VARCHAR(50) = 'John Doe';
PRINT 'Hello, ' + @name; -- Expected output: Hello, John Doe

-- Example 2: Using IF statement
DECLARE @age INT = 25;
IF @age >= 18
    PRINT 'You are an adult.'; -- Expected output: You are an adult.
ELSE
    PRINT 'You are a minor.';

-- Example 3: Creating a stored procedure
CREATE PROCEDURE GetEmployeeCount
AS
BEGIN
    SELECT COUNT(*) AS EmployeeCount FROM Employees;
END;

-- Example 4: Executing a stored procedure
EXEC GetEmployeeCount; -- Expected output: EmployeeCount

# NoSQL (eg. MongoDB)
MongoDB is a popular NoSQL database that provides a flexible and scalable approach to storing and retrieving data. In this code snippet, we demonstrate various operations using MongoDB's syntax.

- We start by creating a collection named "users" using the `createCollection` method.
- Next, we insert a document into the "users" collection using the `insertOne` method.
- To retrieve all documents in the collection, we use the `find` method without any parameters.
- We can also find documents that match a specific condition by passing a query object to the `find` method. In this example, we search for documents where the "age" field is greater than 25.
- Updating a document is done using the `updateOne` method. We specify the filter to identify the document to update and the update operation using the `$set` operator.
- Deleting a document is achieved using the `deleteOne` method. We provide a filter to identify the document to delete.
- Indexes can be created on fields to improve query performance. Here, we create an index on the "name" field using the `createIndex` method.
- Finally, we drop the "users" collection using the `drop` method.

When running this code, you should see the output of the various operations, such as the inserted document, the found documents, and the result of update and delete operations.

In [1]:
-- Create a collection in MongoDB
db.createCollection("users")

-- Insert a document into the collection
db.users.insertOne({name: "John", age: 30})

-- Find all documents in the collection
db.users.find()

-- Find documents matching a specific condition
db.users.find({age: {$gt: 25}})

-- Update a document in the collection
db.users.updateOne({name: "John"}, {$set: {age: 35}})

-- Delete a document from the collection
db.users.deleteOne({name: "John"})

-- Create an index on a field
db.users.createIndex({name: 1})

-- Drop the collection
db.users.drop()

SyntaxError: invalid syntax (2377635783.py, line 1)

# PostgreSQL (Postgres) Dialect

PostgreSQL is a full-featured open-source database whose dialect extends standard SQL in several directions:

- It has *OOP* features the other implementations don't (e.g. table inheritance and composite types), though you usually won't touch them in practice.
- It is more fully featured than MySQL, though MySQL may be faster for read-heavy apps.
- It is a good fit for *complex apps* and *multi-tenant apps*, partly because of the security features shown below.

**Row-Level Security (RLS)** enforces the visibility of rows for database roles. It replaces adding a `WHERE` clause to every backend query, and avoids the leak that happens when that clause gets mangled or left out. It is configured on the table via `ALTER TABLE` and `CREATE POLICY`, and you pick how it will filter — e.g. based on a given column, matched against either the connecting role (which came in via the *connection string*) or a *runtime variable* set at runtime. Although it appears magic/automatic, the policy predicate has a similar performance hit as if you had used the `WHERE` clause on the backend. Note that the role that creates a table is its owner/admin and sees all rows by default; you can force RLS onto it too with extra config, but it's better to make a second, restricted role for the app and keep the owner for maintenance/repair.

**Runtime variables** are key-value pairs you can set at various scopes. For RLS, use transaction scope (`SET LOCAL`), because the other scopes can leak between connections (e.g. through connection pooling). This is the better filter for the multi-tenant case: a single database role connects for every tenant, and the backend sets the tenant ID in a runtime variable per transaction.

**`SECURITY DEFINER` functions** run with the owner's permissions rather than the caller's. Define the privileged work in a function in the database, explicitly grant `EXECUTE` on it to a lower user, and trigger the function from application code while logged in as that user. That pattern allows actions that directly change one table to cascade into actions on privileged tables that require knowing more than just the current org.

In [ ]:
-- Example 1: RLS in a todo app where each user has their own DB role
-- and only sees their own todos

CREATE TABLE todos (
    id SERIAL PRIMARY KEY,
    owner TEXT NOT NULL DEFAULT current_user,
    task TEXT
);

GRANT SELECT, INSERT, UPDATE, DELETE ON todos TO alice, bob;

-- Turn RLS on for the table
ALTER TABLE todos ENABLE ROW LEVEL SECURITY;

-- Filter based on a column: each role only sees rows whose owner matches it
CREATE POLICY user_todos ON todos
    USING (owner = current_user);

-- Connected as alice: only alice's rows come back, with no WHERE clause
-- to mangle or leave out.  (Performance is still as if the WHERE clause
-- were there.)
SELECT * FROM todos;

In [ ]:
-- Example 2: RLS in a multi-tenant app, filtered by a runtime variable

-- A single role (app_user) connects for every tenant, and the backend sets
-- the tenant ID in a runtime variable instead.
CREATE TABLE accounts (
    id SERIAL PRIMARY KEY,
    tenant_id INT NOT NULL,
    balance NUMERIC
);

ALTER TABLE accounts ENABLE ROW LEVEL SECURITY;

-- Filter based on a runtime variable rather than the connecting role
CREATE POLICY tenant_isolation ON accounts
    USING (tenant_id = current_setting('app.tenant_id')::INT);

-- The owner bypasses RLS by default.  You could force it onto the owner too:
--     ALTER TABLE accounts FORCE ROW LEVEL SECURITY;
-- but it's better to keep the owner for maintenance/repair and make a second,
-- restricted role for the app to connect as.
CREATE ROLE app_user LOGIN;
GRANT SELECT, INSERT, UPDATE, DELETE ON accounts TO app_user;

-- Use transaction scope (SET LOCAL) for the runtime variable -- the other
-- scopes can leak between requests when connections are pooled.
BEGIN;
SET LOCAL app.tenant_id = '42';
-- equivalent: SELECT set_config('app.tenant_id', '42', true);
SELECT * FROM accounts; -- only tenant 42's rows
COMMIT;

In [ ]:
-- Example 3: SECURITY DEFINER functions

-- Functions run with the caller's permissions by default; SECURITY DEFINER
-- makes them run with the owner's permissions instead.
-- app_user has no access to audit_log, but deleting a todo through this
-- function still cascades into it.
CREATE TABLE audit_log (
    id SERIAL PRIMARY KEY,
    who TEXT,
    what TEXT,
    at TIMESTAMP DEFAULT now()
);

CREATE FUNCTION delete_todo(todo_id INT) RETURNS VOID
LANGUAGE sql SECURITY DEFINER AS $$
    DELETE FROM todos WHERE id = todo_id;
    -- session_user is the real login; current_user would be the owner here
    INSERT INTO audit_log (who, what)
    VALUES (session_user, 'deleted todo ' || todo_id);
$$;

-- Explicitly grant the lower user the right to call it
REVOKE ALL ON FUNCTION delete_todo(INT) FROM PUBLIC;
GRANT EXECUTE ON FUNCTION delete_todo(INT) TO app_user;

-- Triggered from application code while logged in as app_user:
SELECT delete_todo(7); -- succeeds, and the audit row is written